# Pipeline Smoke Test

This notebook runs a deliberately tiny end-to-end pass through the forecasting and RL pipeline. 
It is meant to catch integration errors before starting the expensive full experiment.


In [1]:
%load_ext autoreload
%autoreload 2


## Imports And Paths


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "hybrid_pipeline").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from hybrid_pipeline.core_pipeline import CONFIG, configure_vast_ai, prepare_data, print_task_summary
from hybrid_pipeline.experiment_runner import (
    run_experiment,
    build_cl_summary,
    build_fwt_details,
    build_forgetting_details,
    print_and_save_comparison_tables,
)
from hybrid_pipeline.trainers import LOGGER, compute_mase


Project root: C:\Users\Syakir\Downloads\Projects\fyp


C:\Users\Syakir\Downloads\Projects\fyp\.venv\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:30: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


## Runtime Setup


In [3]:
DATA_DIR = str(PROJECT_ROOT / "data" / "processed")
OUTPUT_DIR = str(PROJECT_ROOT / "outputs" / "hybrid_smoke_test")

configure_vast_ai(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    require_gpu=False,  # set True on Vast.ai if you want to require CUDA
)


Runtime diagnostics

Python         : 3.11.9

PyTorch        : 2.3.1+cpu

CUDA available : False

CPU cores      : 12

Vast.ai        : NO

Active device  : CPU

Precision      : 32

{'paths': {'demand_csv': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\data\\processed\\demand_forecasting.csv',
  'rl_csv': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\data\\processed\\rl_environment.csv',
  'checkpoints': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\outputs\\hybrid_smoke_test\\checkpoints',
  'results': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\outputs\\hybrid_smoke_test\\results',
  'logs': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\outputs\\hybrid_smoke_test\\logs',
  'plots': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\outputs\\hybrid_smoke_test\\plots'},
 'tasks': [{'task_id': 1,
   'name': 'Baseline_2023_H1',
   'start': '2023-01-01',
   'end': '2023-05-31',
   'regime': 'baseline'},
  {'task_id': 2,
   'name': 'MegaSale_2023',
   'start': '2023-06-01',
   'end': '2023-12-31',
   'regime': 'mega_sale'},
  {'task_id': 3,
   'name': 'Baseline_2024_H1',
   'start': '2024-01-01',
   'end': '2024-05-31',
   'regime': 'baseline'},
  {'task_id': 4,
   'name

## Tiny Smoke-Test Configuration


In [4]:
# Keep this tiny. The goal is correctness, not final metrics.
CONFIG["tasks"] = CONFIG["tasks"][:2]
CONFIG["model_types"] = ["forecasting", "rl"]
CONFIG["cl_methods"] = {
    "forecasting": ["naive", "recall_ewc", "adaptive_drift", "drift_adaptive_replay_ewc"],
    "rl": ["naive", "recall_ewc", "adaptive_drift", "drift_adaptive_replay_ewc"],
}

CONFIG["forecasting"].update({
    "encoder_length": 28,
    "prediction_length": 7,
    "hidden_size": 16,
    "attention_head_size": 1,
    "hidden_continuous_size": 8,
    "batch_size": 64,
    "max_epochs": 1,
    "early_stop_patience": 1,
})

CONFIG["rl"].update({
    "total_timesteps_per_task": 256,
    "eval_episodes": 1,
    "n_steps": 128,
    "batch_size": 64,
    "n_epochs": 1,
    "net_arch": [32, 32],
})

CONFIG["cl"].update({
    "ewc_fisher_samples": 2,
    "replay_buffer_size": 128,
    "recall_buffer_capacity": 256,
    "recall_mix_n_steps": 32,
})

CONFIG["hardware"].update({
    "compile": False,
    "num_workers": 0,
    "persistent_workers": False,
})

print("Smoke-test config ready")
print("Tasks:", [t["name"] for t in CONFIG["tasks"]])
print("Forecast methods:", CONFIG["cl_methods"]["forecasting"])
print("RL methods:", CONFIG["cl_methods"]["rl"])


Smoke-test config ready
Tasks: ['Baseline_2023_H1', 'MegaSale_2023']
Forecast methods: ['naive', 'recall_ewc', 'adaptive_drift', 'drift_adaptive_replay_ewc']
RL methods: ['naive', 'recall_ewc', 'adaptive_drift', 'drift_adaptive_replay_ewc']


## Metric Sanity Check


In [5]:
mase_value = compute_mase([2, 3, 4], [2, 2, 5], list(range(20)), seasonality=7)
assert mase_value == mase_value and mase_value > 0, mase_value
print("MASE sanity check:", mase_value)


MASE sanity check: 0.09523809523809523


## Load Data


In [6]:
tft_tasks, rl_tasks, tft_df, rl_df = prepare_data()
print_task_summary(tft_tasks, rl_tasks)

assert len(tft_tasks) == len(CONFIG["tasks"])
assert len(rl_tasks) == len(CONFIG["tasks"])
assert all(len(df) > 0 for df in tft_tasks), "At least one TFT task is empty"
assert all(len(df) > 0 for df in rl_tasks), "At least one RL task is empty"
print("Data checks passed")


Loading datasets...

Demand CSV  : 9,864 rows × 24 cols

RL CSV      : 9,864 rows × 19 cols

✓ Data loaded and cleaned

┏━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┓
┃ Task ┃ Name             ┃ Period                   ┃ TFT rows ┃ RL rows ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━┩
│ 1    │ Baseline_2023_H1 │ 2023-01-01 -> 2023-05-31 │ 1,359    │ 1,359   │
│ 2    │ MegaSale_2023    │ 2023-06-01 -> 2023-12-31 │ 1,926    │ 1,926   │
└──────┴──────────────────┴──────────────────────────┴──────────┴─────────┘

Data checks passed


## Run Smoke Test


In [7]:
run_experiment(tft_tasks, rl_tasks)
print("Smoke-test training loop completed")


==============================================================

  CONTINUAL LEARNING EXPERIMENT START

==============================================================

═══ MODEL TYPE: FORECASTING ═══

  ── CL Method: naive ──

...

Eval task 1: avg_episode_reward=5843.0336  cumulative_profit=46902227.5936  pricing_regret=31.9254

Eval task 2: avg_episode_reward=7408.4361  cumulative_profit=43728020.7899  pricing_regret=21.9939

Results saved → C:\Users\Syakir\Downloads\Projects\fyp\outputs\hybrid_smoke_test\results\all_metrics.csv

═══ EXPERIMENT COMPLETE (2.9 min) ═══

Smoke-test training loop completed


## Validate Results


In [8]:
results_df = LOGGER.to_dataframe()
display(results_df.tail(20))

assert not results_df.empty, "No metrics were logged"
expected_model_types = set(CONFIG["model_types"])
assert expected_model_types.issubset(set(results_df["model_type"])), results_df["model_type"].unique()

forecast_df = results_df[results_df["model_type"] == "forecasting"]
rl_df_results = results_df[results_df["model_type"] == "rl"]
assert not forecast_df.empty, "No forecasting metrics logged"
assert not rl_df_results.empty, "No RL metrics logged"

# MASE can be NaN if a deliberately tiny smoke split has insufficient scale,
# but sMAPE/RMSE and RL metrics should exist.
assert {"smape", "rmse"}.issubset(set(forecast_df["metric_name"])), forecast_df["metric_name"].unique()
assert "cumulative_profit" in set(rl_df_results["metric_name"]), rl_df_results["metric_name"].unique()

print("Logged metrics:")
print(results_df.groupby(["model_type", "cl_method", "metric_name"]).size())
print("Smoke test passed")


,model_type,cl_method,train_task_id,eval_task_id,eval_phase,metric_name,metric_value,timestamp
92,rl,drift_adaptive_replay_ewc,2,1,seen,pricing_regret,3.192541e+01,2026-06-03T22:02:29
93,rl,drift_adaptive_replay_ewc,2,2,seen,avg_episode_reward,7.408436e+03,2026-06-03T22:02:29
94,rl,drift_adaptive_replay_ewc,2,2,seen,cumulative_profit,4.372802e+07,2026-06-03T22:02:29
95,rl,drift_adaptive_replay_ewc,2,2,seen,pricing_regret,2.199385e+01,2026-06-03T22:02:29
96,rl,naive,1,1,seen,profit_index,1.000000e+00,2026-06-03T22:02:30
97,rl,naive,1,2,future,profit_index,1.003766e+00,2026-06-03T22:02:30
98,rl,naive,2,1,seen,profit_index,9.835561e-01,2026-06-03T22:02:30
99,rl,naive,2,2,seen,profit_index,1.000000e+00,2026-06-03T22:02:30
100,rl,recall_ewc,1,1,seen,profit_index,1.000000e+00,2026-06-03T22:02:30
101,rl,recall_ewc,1,2,future,profit_index,1.003766e+00,2026-06-03T22:02:30


Logged metrics:
model_type   cl_method                  metric_name       
forecasting  adaptive_drift             mase                  4
                                        rmse                  4
                                        smape                 4
             drift_adaptive_replay_ewc  mase                  4
                                        rmse                  4
                                        smape                 4
             naive                      mase                  4
                                        rmse                  4
                                        smape                 4
             recall_ewc                 mase                  4
                                        rmse                  4
                                        smape                 4
rl           adaptive_drift             avg_episode_reward    4
                                        cumulative_profit     4
                             

## Optional Summary Tables


In [9]:
cl_summary = build_cl_summary()
fwt_details = build_fwt_details(save=False)
forgetting_details = build_forgetting_details(save=False)
tables = print_and_save_comparison_tables(cl_summary)
cl_summary


FWT details saved -> C:\Users\Syakir\Downloads\Projects\fyp\outputs\hybrid_smoke_test\results\fwt_details.csv

Forgetting details saved -> 
C:\Users\Syakir\Downloads\Projects\fyp\outputs\hybrid_smoke_test\results\forgetting_details.csv

CL Summary (BWT / FWT / Forgetting):

model_type                 cl_method primary_metric  avg_final_perf  avg_online_perf     bwt     fwt  forgetting  
avg_forgetting
forecasting                     naive           mase          8.2106           8.8416  1.2620  0.0000      0.0000  
0.0000
forecasting                recall_ewc           mase          2.5687           2.5688  0.0001 21.3267      0.0000  
0.0000
forecasting            adaptive_drift           mase          7.1128           7.1185  0.0113 12.3727      0.0000  
0.0000
forecasting drift_adaptive_replay_ewc           mase          5.3483           5.3323 -0.0320 16.9661      0.0320  
0.0320
         rl                     naive   profit_index          0.9918           1.0000 -0.0164  0.0000      0.0164  
0.0164
         rl                recall_ewc   profit_index          0.9771           0.9515  0.0514  0.0000      0.0000  
0.0000
         rl            adaptive_drift   profit_index          1.0337           1.0282  0.0110  0.0000      0.0000  
0.0000
         rl drift_adaptive_replay_ewc   profit_index          1.0285           0.9774  0.1022  0.0000      0.0000  
0.0000


  FORECASTING - MASE (lower is better)
                           Task 1   Task 2
cl_method                                 
adaptive_drift             2.0337  12.2033
drift_adaptive_replay_ewc  1.8550   8.8096
naive                      3.1013  14.5819
recall_ewc                 1.1085   4.0290

  FORECASTING - sMAPE
                             Task 1    Task 2
cl_method                                    
adaptive_drift              85.6333  102.8248
drift_adaptive_replay_ewc   90.3793   93.7928
naive                      107.8053  110.5400
recall_ewc                 120.1402  119.4468

  RL - PROFIT INDEX (vs naive fresh per-task, higher is better)
                           Task 1  Task 2
cl_method                                
adaptive_drift                1.0  1.0565
drift_adaptive_replay_ewc     1.0  0.9548
naive                         1.0  1.0000
recall_ewc                    1.0  0.9029

  RL - CUMULATIVE PROFIT (raw MYR, reference)
                                 Task 1

,model_type,cl_method,primary_metric,avg_final_perf,avg_online_perf,bwt,fwt,forgetting,avg_forgetting
0,forecasting,naive,mase,8.2106,8.8416,1.2620,0.0000,0.0000,0.0000
1,forecasting,recall_ewc,mase,2.5687,2.5688,0.0001,21.3267,0.0000,0.0000
2,forecasting,adaptive_drift,mase,7.1128,7.1185,0.0113,12.3727,0.0000,0.0000
3,forecasting,drift_adaptive_replay_ewc,mase,5.3483,5.3323,-0.0320,16.9661,0.0320,0.0320
4,rl,naive,profit_index,0.9918,1.0000,-0.0164,0.0000,0.0164,0.0164
5,rl,recall_ewc,profit_index,0.9771,0.9515,0.0514,0.0000,0.0000,0.0000
6,rl,adaptive_drift,profit_index,1.0337,1.0282,0.0110,0.0000,0.0000,0.0000
7,rl,drift_adaptive_replay_ewc,profit_index,1.0285,0.9774,0.1022,0.0000,0.0000,0.0000
